In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import json
from pathlib import Path
from tqdm import tqdm
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import json
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import json
from pathlib import Path
from tqdm import tqdm

# HANYA ini yang perlu untuk MediaPipe
import mediapipe as mp

# Inisialisasi Face Mesh
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

# Create face mesh instance
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("✓ MediaPipe Face Mesh ready")
# (Not using MediaPipe yet - we'll add later for mouth ROI extraction)

class LipReadingDataset(Dataset):
    def __init__(self, data_dir="datasets", sequence_length=30, augment=False):
        """
        Args:
            data_dir: Root directory containing word folders
            sequence_length: Frames per sequence (pad atau truncate ke ini)
            augment: Enable data augmentation
        """
        self.data_dir = data_dir
        self.sequence_length = sequence_length
        self.augment = augment
        self.samples = []
        self.word2idx = {}
        self.idx2word = {}
        
        # Build sample list
        self._build_dataset()
        
    def _build_dataset(self):
        """Scan datasets directory dan build samples"""
        word_idx = 0
        for word_folder in sorted(os.listdir(self.data_dir)):
            word_path = os.path.join(self.data_dir, word_folder)
            if not os.path.isdir(word_path):
                continue
            
            self.word2idx[word_folder] = word_idx
            self.idx2word[word_idx] = word_folder
            word_idx += 1
            
            # Iterate speakers
            for speaker_folder in os.listdir(word_path):
                speaker_path = os.path.join(word_path, speaker_folder)
                frames_path = os.path.join(speaker_path, "raw", "frames")
                
                if os.path.isdir(frames_path) and len(os.listdir(frames_path)) > 0:
                    self.samples.append({
                        'word': word_folder,
                        'word_idx': self.word2idx[word_folder],
                        'frames_path': frames_path,
                        'speaker': speaker_folder
                    })
    def _extract_mouth_roi(self, frame, face_mesh_results):
        """
        Extract mouth ROI dari frame menggunakan MediaPipe landmarks
        
        Mouth landmarks (MediaPipe Face Mesh):
        - Upper lip: 61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291
        - Lower lip: 78, 81, 82, 13, 312, 311, 310, 415, 308, 325, 307
        """
        if face_mesh_results.multi_face_landmarks is None or len(face_mesh_results.multi_face_landmarks) == 0:
            return None
        
        landmarks = face_mesh_results.multi_face_landmarks[0].landmark
        h, w, c = frame.shape
        
        # Mouth landmark indices (bibir atas dan bibir bawah)
        mouth_landmarks = [61, 40, 39, 37, 0, 267, 269, 270, 409, 291, 
                        78, 81, 82, 13, 312, 311, 310, 415, 308, 325, 307]
        
        # Get bounding box dari mouth region
        mouth_coords = []
        for idx in mouth_landmarks:
            lm = landmarks[idx]
            x, y = int(lm.x * w), int(lm.y * h)
            mouth_coords.append([x, y])
        
        mouth_coords = np.array(mouth_coords)
        x_min, y_min = mouth_coords.min(axis=0)
        x_max, y_max = mouth_coords.max(axis=0)
        
        # Add padding
        padding = 10
        x_min = max(0, x_min - padding)
        y_min = max(0, y_min - padding)
        x_max = min(w, x_max + padding)
        y_max = min(h, y_max + padding)
        
        # Extract ROI
        mouth_roi = frame[y_min:y_max, x_min:x_max]
        
        if mouth_roi.size == 0:
            return None
        
        # Resize to fixed size (96x96 untuk mouth area)
        mouth_roi = cv2.resize(mouth_roi, (96, 96))
        return mouth_roi 
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        frames_path = sample['frames_path']
        word_idx = sample['word_idx']
        
        # Load frames
        frame_files = sorted([f for f in os.listdir(frames_path) if f.endswith('.jpg')])
        mouth_rois = []
        
        for frame_file in frame_files[:self.sequence_length]:
            frame_path = os.path.join(frames_path, frame_file)
            frame = cv2.imread(frame_path)
            if frame is None:
                continue
            
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Extract mouth ROI using MediaPipe
            face_results = face_mesh.process(frame)
            mouth_roi = self._extract_mouth_roi(frame, face_results)
            
            if mouth_roi is not None:
                mouth_rois.append(mouth_roi)
        
        # Pad jika kurang frames
        if len(mouth_rois) < self.sequence_length:
            last_roi = mouth_rois[-1] if mouth_rois else np.zeros((96, 96, 3), dtype=np.uint8)
            while len(mouth_rois) < self.sequence_length:
                mouth_rois.append(last_roi)
        
        # Convert to tensor (C, T, H, W)
        mouth_rois = np.stack(mouth_rois[:self.sequence_length])  # (T, 96, 96, 3)
        mouth_rois = torch.FloatTensor(mouth_rois).permute(3, 0, 1, 2) / 255.0  # (C, T, 96, 96)
        
        return {
            'frames': mouth_rois,  # Now mouth ROI instead of full frame
            'label': word_idx,
            'word': sample['word']
        }

# Test dataset
print("Loading dataset...")
dataset = LipReadingDataset(sequence_length=30)
print(f"✓ Total samples: {len(dataset)}")
print(f"✓ Word classes: {dataset.word2idx}")

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
batch = next(iter(loader))
print(f"✓ Batch shape: {batch['frames'].shape}")  # (B, C, T, H, W)

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [ ]:
class LipReading3DCNN(nn.Module):
    """
    3D CNN + LSTM untuk Lip Reading
    
    Architecture:
    - 3 layers 3D CNN (spatial-temporal convolution)
    - 2 layers LSTM (temporal sequence modeling)
    - Dense layer (classification)
    """
    
    def __init__(self, num_classes, num_lstm_layers=2, hidden_dim=256):
        super().__init__()
        
        # 3D Convolution layers
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 5, 5), padding=(1, 2, 2))
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        
        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 5, 5), padding=(1, 2, 2))
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        
        self.conv3 = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn3 = nn.BatchNorm3d(256)
        self.pool3 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        
        self.dropout = nn.Dropout(0.5)
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=256 * 12 * 12,  # Updated from 256 * 14 * 14
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.5 if num_lstm_layers > 1 else 0
        )
        
        # Classification head
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
        
    def forward(self, x):
        """
        Args:
            x: (B, C, T, H, W)
        Returns:
            logits: (B, T, num_classes) - untuk CTC compatibility
        """
        B, C, T, H, W = x.shape
        
        # 3D Conv blocks
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool2(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = torch.relu(x)
        x = self.pool3(x)
        
        # Flatten spatial dims, keep temporal
        x = x.permute(0, 2, 1, 3, 4)  # (B, T, C, H, W)
        x = x.reshape(B, T, -1)  # (B, T, C*H*W)
        
        # LSTM
        x, _ = self.lstm(x)  # (B, T, hidden_dim*2)
        x = self.dropout(x)
        
        # Dense classification
        x = self.fc(x)  # (B, T, num_classes)
        
        return x

# Initialize model
num_classes = len(dataset.word2idx)
model = LipReading3DCNN(num_classes=num_classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"✓ Model initialized on {device}")
print(f"✓ Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# CTC Loss + Optimizer setup
ctc_loss = nn.CTCLoss(blank=num_classes, reduction='mean', zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

def train_epoch(model, train_loader, optimizer, device):
    """Train satu epoch"""
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc="Training"):
        frames = batch['frames'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        logits = model(frames)  # (B, T, num_classes)
        
        # CTC Loss perlu shapes tertentu:
        # logits: (T, B, num_classes)
        # predicted_lengths: (B,) - banyak timesteps
        # target_lengths: (B,) - panjang target (1 untuk single word per sample)
        
        logits_t = logits.permute(1, 0, 2)  # (T, B, num_classes)
        
        # Assuming satu kata per sample (target length = 1)
        target_lengths = torch.ones(labels.shape[0], dtype=torch.long, device=device)
        input_lengths = torch.full(
            (logits_t.shape[1],), 
            logits_t.shape[0], 
            dtype=torch.long, 
            device=device
        )
        
        loss = ctc_loss(logits_t, labels.unsqueeze(1), input_lengths, target_lengths)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def validate(model, val_loader, device):
    """Validation loop"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            frames = batch['frames'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(frames)  # (B, T, num_classes)
            
            # Get predicted class (max logit across time)
            preds = logits.max(dim=2)[1].max(dim=1)[0]  # (B,)
            
            correct += (preds == labels).sum().item()
            total += labels.shape[0]
    
    accuracy = correct / total
    return accuracy

print("✓ CTC Loss & training setup ready")

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset
train_idx, val_idx = train_test_split(
    range(len(dataset)), 
    test_size=0.2, 
    random_state=42,
    stratify=[s['word_idx'] for s in dataset.samples]
)

train_subset = torch.utils.data.Subset(dataset, train_idx)
val_subset = torch.utils.data.Subset(dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=8, shuffle=False, num_workers=0)

print(f"✓ Train samples: {len(train_subset)}")
print(f"✓ Val samples: {len(val_subset)}")

# Training loop
num_epochs = 50
best_val_acc = 0
patience_counter = 0
patience = 10

history = {'train_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1:3d}/{num_epochs}")
    print(f"{'='*60}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, device)
    history['train_loss'].append(train_loss)
    print(f"Train Loss: {train_loss:.4f}")
    
    # Validate
    val_acc = validate(model, val_loader, device)
    history['val_acc'].append(val_acc)
    print(f"Val Accuracy: {val_acc:.4f}")
    
    # Learning rate scheduling
    scheduler.step(train_loss)
    
    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_lipreading_model.pth')
        print(f"✓ Model saved (Best: {best_val_acc:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")
    
    if patience_counter >= patience:
        print(f"\n✓ Early stopping at epoch {epoch+1}")
        break

print(f"\n✓ Training complete! Best val accuracy: {best_val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

# Load best model & evaluate on test set
model.load_state_dict(torch.load('best_lipreading_model.pth'))
test_acc = validate(model, val_loader, device)
print(f"✓ Test Accuracy: {test_acc:.4f}")

# Inference function
def predict_word(frames_dir, model, dataset):
    """
    Predict word dari folder frames
    
    Args:
        frames_dir: Path ke folder frames
        model: Trained model
        dataset: Dataset instance (untuk word2idx mapping)
    """
    frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
    frames = []
    
    for frame_file in frame_files[:30]:
        frame_path = os.path.join(frames_dir, frame_file)
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    
    if len(frames) < 30:
        last_frame = frames[-1]
        while len(frames) < 30:
            frames.append(last_frame)
    
    frames = np.stack(frames)
    frames = torch.FloatTensor(frames).permute(3, 0, 1, 2) / 255.0
    frames = torch.nn.functional.interpolate(
        frames.unsqueeze(0), size=(112, 112), mode='trilinear'
    ).squeeze(0)
    frames = frames.unsqueeze(0).to(device)
    
    with torch.no_grad():
        model.eval()
        logits = model(frames)
        pred_idx = logits.max(dim=2)[1].max(dim=1)[0].item()
        confidence = logits.max(dim=2)[0].mean(dim=1).max().item()
    
    predicted_word = dataset.idx2word[pred_idx]
    return predicted_word, confidence

# Test inference
test_sample = dataset.samples[0]
pred_word, conf = predict_word(test_sample['frames_path'], model, dataset)
print(f"\nPredicted word: {pred_word} (confidence: {conf:.4f})")
print(f"Actual word: {test_sample['word']}")